Imports

In [22]:
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import pandas as pd
import numpy as np

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from sklearn.preprocessing import StandardScaler


Load data

In [23]:
df = pd.read_csv("ppg_bp_features_extended.csv")

# Encode gender if necessary
df['Gender'] = df['Gender'].map({'Male': 0, 'Female': 1})

# Features and targets
X = df.drop(columns=['SBP', 'DBP', 'HR', 'Subject', 'Segment'])
y = df[['SBP', 'DBP', 'HR']]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Helper Function

In [24]:
def train_evaluate_model(model, X_train, X_test, y_train, y_test, name="Model"):
    multi_model = MultiOutputRegressor(model)
    multi_model.fit(X_train, y_train)
    y_pred = multi_model.predict(X_test)

    results = {}
    overall_mae = []

    for i, col in enumerate(y_train.columns):
        mae = mean_absolute_error(y_test[col], y_pred[:, i])
        r2 = r2_score(y_test[col], y_pred[:, i])
        results[col] = {"MAE": mae, "R2": r2}
        overall_mae.append(mae)

    results['Overall'] = {"MAE": np.mean(overall_mae)}
    print(f"\n{name} Results:")
    for k, v in results.items():
        if k != 'Overall':
            print(f"{k} → MAE: {v['MAE']:.2f}, R²: {v['R2']:.3f}")
    print(f"✅ Overall MAE: {results['Overall']['MAE']:.2f}")

    return multi_model, results


Train and evaulate Model

In [25]:
rf_model, rf_results = train_evaluate_model(RandomForestRegressor(n_estimators=200, random_state=42),
                                            X_train, X_test, y_train, y_test, name="Random Forest")

xgb_model, xgb_results = train_evaluate_model(XGBRegressor(n_estimators=200, random_state=42),
                                              X_train, X_test, y_train, y_test, name="XGBoost")

lgb_model, lgb_results = train_evaluate_model(LGBMRegressor(n_estimators=200, random_state=42),
                                              X_train, X_test, y_train, y_test, name="LightGBM")



Random Forest Results:
SBP → MAE: 11.66, R²: 0.435
DBP → MAE: 7.37, R²: 0.299
HR → MAE: 3.72, R²: 0.692
✅ Overall MAE: 7.58

XGBoost Results:
SBP → MAE: 10.63, R²: 0.535
DBP → MAE: 6.71, R²: 0.381
HR → MAE: 4.02, R²: 0.622
✅ Overall MAE: 7.12
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000155 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2058
[LightGBM] [Info] Number of data points in the train set: 525, number of used features: 19
[LightGBM] [Info] Start training from score 127.902857
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits 

Stacking: Tree + Deep Learning

In [29]:
# Only use tree predictions as input
X_train_stack = X_train_tree_preds
X_test_stack  = X_test_tree_preds

# Scale features
scaler = StandardScaler()
X_train_stack_scaled = scaler.fit_transform(X_train_stack)
X_test_stack_scaled  = scaler.transform(X_test_stack)

# Smaller neural network with dropout
nn_model = Sequential([
    Dense(32, activation='relu', input_shape=(X_train_stack_scaled.shape[1],)),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(3)  # SBP, DBP, HR
])

# Lower learning rate
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
nn_model.compile(optimizer=optimizer, loss='mean_absolute_error')

# Early stopping
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)

# Train
history = nn_model.fit(
    X_train_stack_scaled, y_train,
    validation_split=0.2,
    epochs=150,
    batch_size=16,
    callbacks=[early_stop],
    verbose=1
)

# Evaluate
y_pred_stack = nn_model.predict(X_test_stack_scaled)
mae_sbp = mean_absolute_error(y_test['SBP'], y_pred_stack[:,0])
mae_dbp = mean_absolute_error(y_test['DBP'], y_pred_stack[:,1])
mae_hr  = mean_absolute_error(y_test['HR'], y_pred_stack[:,2])
overall_mae = np.mean([mae_sbp, mae_dbp, mae_hr])

print(f"\nStacked Model Results (Adjusted):")
print(f"SBP MAE: {mae_sbp:.2f}")
print(f"DBP MAE: {mae_dbp:.2f}")
print(f"HR  MAE: {mae_hr:.2f}")
print(f"✅ Overall MAE: {overall_mae:.2f}")


Epoch 1/150


c:\Users\user\miniconda3\envs\newenv\Lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 90.5081 - val_loss: 91.3238
Epoch 2/150
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 89.8214 - val_loss: 90.4327
Epoch 3/150
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 88.4646 - val_loss: 88.6825
Epoch 4/150
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 85.9950 - val_loss: 85.3278
Epoch 5/150
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 81.6602 - val_loss: 79.6627
Epoch 6/150
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 74.1938 - val_loss: 70.8467
Epoch 7/150
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 64.4218 - val_loss: 59.6165
Epoch 8/150
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 52.8719 - val_loss: 48.1238
Epoch 9/150
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 43.7032 - val_loss: 40.1154
Epoch 10/150
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 39.2233 - val_loss: 35.1938
Epoch 11/150
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 35.0570 - val_loss: 32.0002
Epoch 12/150
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step -

In [30]:
# %% [markdown]
# Weighted Stacked Model + Residual NN

# %% 
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
import numpy as np

# --- 1. Calculate tree model weights based on inverse MAE ---
# Using your previous results
mae_list = np.array([
    rf_results['Overall']['MAE'],
    xgb_results['Overall']['MAE'],
    lgb_results['Overall']['MAE']
])

weights = 1 / mae_list
weights /= weights.sum()  # normalize to sum 1
print("Tree Weights:", weights)

# --- 2. Weighted tree predictions ---
def weighted_preds(models, weights, X):
    preds = np.hstack([m.predict(X) for m in models])  # shape (n_samples, 3*num_models)
    weighted = np.zeros((X.shape[0], 3))
    for i, w in enumerate(weights):
        weighted += preds[:, i*3:(i+1)*3] * w
    return weighted

tree_models = [rf_model, xgb_model, lgb_model]

X_train_weighted = weighted_preds(tree_models, weights, X_train)
X_test_weighted  = weighted_preds(tree_models, weights, X_test)

# --- 3. Residuals for training ---
y_train_residuals = y_train.values - X_train_weighted

# --- 4. Scale weighted predictions for NN ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_weighted)
X_test_scaled  = scaler.transform(X_test_weighted)

# --- 5. Build small NN to predict residuals ---
nn_residual = Sequential([
    Dense(32, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(3)  # residuals for SBP, DBP, HR
])

optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
nn_residual.compile(optimizer=optimizer, loss='mean_absolute_error')

# Early stopping
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)

# Train
history = nn_residual.fit(
    X_train_scaled, y_train_residuals,
    validation_split=0.2,
    epochs=150,
    batch_size=16,
    callbacks=[early_stop],
    verbose=1
)

# --- 6. Final predictions ---
residual_preds = nn_residual.predict(X_test_scaled)
y_pred_final = X_test_weighted + residual_preds  # add residuals

# --- 7. Evaluate ---
mae_sbp = mean_absolute_error(y_test['SBP'], y_pred_final[:,0])
mae_dbp = mean_absolute_error(y_test['DBP'], y_pred_final[:,1])
mae_hr  = mean_absolute_error(y_test['HR'], y_pred_final[:,2])
overall_mae = np.mean([mae_sbp, mae_dbp, mae_hr])

print(f"\nWeighted Stacked + Residual NN Results:")
print(f"SBP MAE: {mae_sbp:.2f}")
print(f"DBP MAE: {mae_dbp:.2f}")
print(f"HR  MAE: {mae_hr:.2f}")
print(f"✅ Overall MAE: {overall_mae:.2f}")


Tree Weights: [0.32712203 0.34848969 0.32438828]
Epoch 1/150


c:\Users\user\miniconda3\envs\newenv\Lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 1.0076 - val_loss: 1.1099
Epoch 2/150
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.9405 - val_loss: 1.0383
Epoch 3/150
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.8996 - val_loss: 0.9821
Epoch 4/150
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.8501 - val_loss: 0.9249
Epoch 5/150
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.8152 - val_loss: 0.8712
Epoch 6/150
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.7827 - val_loss: 0.8338
Epoch 7/150
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.7637 - val_loss: 0.8070
Epoch 8/150
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.7657 - val_loss: 0.7965
Epoch 9/150
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.7480 - val_loss: 0.7771
Epoch 10/150
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.7421 - val_loss: 0.7859
Epoch 11/150
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.7342 - val_loss: 0.7707
Epoch 12/150
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.7230 - val_lo